# Data Leakage: `fit` Sebelum Split vs Sesudah Split

Notebook ini memakai data sintetis **10 baris, 2 fitur numerik** — supaya setiap angka
masih bisa dihitung manual di slide.

Aturan intinya:

> Semua yang **belajar dari data** (scaler, encoder, imputer, feature selection)
> harus di-`fit` **hanya pada data train**, lalu dipakai untuk men-`transform` data test.

| Bagian | Isi |
|---|---|
| 1. Data sintetis | Data apa yang dipakai |
| 2. ❌ Cara salah | `fit_transform` seluruh data, baru di-split |
| 3. ✅ Cara benar | Split dulu, `fit` di train, `transform` test |
| 4. Perbandingan | Selisih angka hasil kedua cara |
| 5. StandardScaler | Kebocoran yang sama pada mean & std |
| 6. Missing value | Kebocoran pada imputasi |
| 7. Efek ke skor model | Kenapa ini berbahaya |
| 8. Solusi: `Pipeline` | Cara aman yang dipakai di praktik |

## 1. Data sintetis (10 baris, 2 fitur)

Kasus: memprediksi **tingkat risiko kredit** pemohon pinjaman.

| Kolom | Keterangan |
|---|---|
| `income` | pendapatan per tahun (ribu USD) |
| `credit_score` | skor kredit (300-850) |
| `risk` | **target** — Low / Medium / High |

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

df = pd.DataFrame({
    'id':           [f'P{i:02d}' for i in range(1, 11)],
    'income':       [28, 35, 42, 48, 55, 63, 78, 95, 130, 150],
    'credit_score': [580, 610, 640, 660, 690, 710, 730, 760, 790, 820],
    'risk':         ['High', 'High', 'Medium', 'Medium', 'Medium',
                     'Low', 'Low', 'Low', 'Low', 'Low'],
})
df

,id,income,credit_score,risk
0,P01,28,580,High
1,P02,35,610,High
2,P03,42,640,Medium
3,P04,48,660,Medium
4,P05,55,690,Medium
5,P06,63,710,Low
6,P07,78,730,Low
7,P08,95,760,Low
8,P09,130,790,Low
9,P10,150,820,Low


In [7]:
FEATURES = ['income', 'credit_score']

X = df[FEATURES]
y = df['risk']

RANDOM_STATE = 9   # dikunci supaya split-nya sama persis di kedua cara
TEST_SIZE = 0.3    # 7 baris train, 3 baris test

## 2. ❌ Cara salah: `fit_transform` seluruh data, baru di-split

Ini pola yang sering muncul karena terasa "rapi": semua data diskalakan sekaligus,
baru dibagi. Masalahnya, `MinMaxScaler` menghitung **min dan max dari 10 baris** —
termasuk 3 baris yang nanti jadi test set.

In [8]:
scaler_salah = MinMaxScaler()
X_scaled_all = pd.DataFrame(
    scaler_salah.fit_transform(X),      # <-- fit pada SELURUH data
    columns=FEATURES, index=X.index,
).round(3)

print('min yang dipelajari scaler:', scaler_salah.data_min_)
print('max yang dipelajari scaler:', scaler_salah.data_max_)

X_train_salah, X_test_salah, y_train, y_test = train_test_split(
    X_scaled_all, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

min yang dipelajari scaler: [ 28. 580.]
max yang dipelajari scaler: [150. 820.]


In [10]:
X_scaled_all

,income,credit_score
0,0.000,0.000
1,0.057,0.125
2,0.115,0.250
3,0.164,0.333
4,0.221,0.458
5,0.287,0.542
6,0.410,0.625
7,0.549,0.750
8,0.836,0.875
9,1.000,1.000


In [9]:
X_train_salah

,income,credit_score
6,0.410,0.625
7,0.549,0.750
1,0.057,0.125
5,0.287,0.542
8,0.836,0.875
4,0.221,0.458
3,0.164,0.333


In [6]:
X_test_salah

,income,credit_score
9,1.000,1.00
2,0.115,0.25
0,0.000,0.00


In [4]:
test_salah = df.loc[X_test_salah.index, ['id', 'income', 'credit_score']].join(
    X_test_salah.add_suffix('_scaled')
)
test_salah

,id,income,credit_score,income_scaled,credit_score_scaled
9,P10,150,820,1.000,1.00
2,P03,42,640,0.115,0.25
0,P01,28,580,0.000,0.00


Perhatikan barisnya: **P10 tepat 1.000 dan P01 tepat 0.000**.

Nilai 0 dan 1 hanya muncul pada baris yang menjadi min/max saat `fit`. Artinya scaler
tahu bahwa 150 adalah nilai tertinggi dan 28 adalah nilai terendah — padahal kedua baris
itu ada di **test set**, data yang seharusnya belum pernah dilihat.
Test set jadi terlihat "terlalu pas", dan evaluasi model jadi terlalu optimis.

## 3. ✅ Cara benar: split dulu, `fit` di train, `transform` di test

Urutannya dibalik. Scaler hanya melihat 7 baris train, sehingga min/max yang dipelajari
berbeda — dan itu memang yang akan terjadi di dunia nyata, di mana data baru belum ada
saat model dilatih.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler_benar = MinMaxScaler()
X_train_benar = pd.DataFrame(
    scaler_benar.fit_transform(X_train),   # <-- fit HANYA pada data train
    columns=FEATURES, index=X_train.index,
).round(3)

X_test_benar = pd.DataFrame(
    scaler_benar.transform(X_test),        # <-- test hanya di-transform, tidak di-fit
    columns=FEATURES, index=X_test.index,
).round(3)

print('min yang dipelajari scaler:', scaler_benar.data_min_)
print('max yang dipelajari scaler:', scaler_benar.data_max_)

min yang dipelajari scaler: [ 35. 610.]
max yang dipelajari scaler: [130. 790.]


In [6]:
test_benar = df.loc[X_test_benar.index, ['id', 'income', 'credit_score']].join(
    X_test_benar.add_suffix('_scaled')
)
test_benar

,id,income,credit_score,income_scaled,credit_score_scaled
9,P10,150,820,1.211,1.167
2,P03,42,640,0.074,0.167
0,P01,28,580,-0.074,-0.167


Sekarang **P10 bernilai 1.211 (di atas 1) dan P01 bernilai -0.074 (di bawah 0)**.

Ini bukan bug: hasil di luar rentang [0, 1] justru **wajar dan sehat**. Test set memang
berisi nilai yang lebih ekstrem daripada apa pun yang pernah dilihat model saat latihan —
persis seperti data baru yang datang setelah model dipakai di produksi.

## 4. Perbandingan angka test set

In [7]:
perbandingan = pd.DataFrame({
    'id':                df.loc[X_test.index, 'id'],
    'income (asli)':     df.loc[X_test.index, 'income'],
    'SALAH: fit semua data': X_test_salah['income'],
    'BENAR: fit train saja': X_test_benar['income'],
}).set_index('id')
perbandingan['selisih'] = (perbandingan['BENAR: fit train saja']
                           - perbandingan['SALAH: fit semua data']).round(3)
perbandingan

,income (asli),SALAH: fit semua data,BENAR: fit train saja,selisih
id,,,,
P10,150,1.000,1.211,0.211
P03,42,0.115,0.074,-0.041
P01,28,0.000,-0.074,-0.074


Angka yang masuk ke model berbeda untuk **setiap baris test**. Model yang sama, data yang
sama, tapi skor evaluasinya tidak bisa dibandingkan — versi kiri sudah "dibantu" oleh
informasi dari test set.

## 5. Kebocoran yang sama pada `StandardScaler`

`StandardScaler` belajar **mean** dan **std**. Kalau di-`fit` pada seluruh data, kedua
statistik itu ikut menyerap nilai-nilai dari test set.

In [8]:
stat = pd.DataFrame({
    'SALAH: fit semua data (10 baris)': StandardScaler().fit(X).mean_,
    'BENAR: fit train saja (7 baris)':  StandardScaler().fit(X_train).mean_,
}, index=FEATURES).round(2)

std = pd.DataFrame({
    'SALAH: fit semua data (10 baris)': StandardScaler().fit(X).scale_,
    'BENAR: fit train saja (7 baris)':  StandardScaler().fit(X_train).scale_,
}, index=FEATURES).round(2)

statistik = pd.concat({'mean': stat, 'std': std})
statistik

SALAH: fit semua data (10 baris)  \
mean income                                   72.40   
     credit_score                            699.00   
std  income                                   38.94   
     credit_score                             74.09   

                   BENAR: fit train saja (7 baris)  
mean income                                  72.00  
     credit_score                           707.14  
std  income                                  29.87  
     credit_score                            56.24

`std` untuk `income` melonjak dari 29.87 menjadi 38.94 hanya karena nilai 150 (baris test)
ikut dihitung. Semua fitur hasil standardisasi jadi bergeser.

## 6. Kebocoran pada imputasi missing value

Aturannya sama untuk `SimpleImputer`: nilai pengisi (mean/median/modus) harus dihitung
dari data train saja. Di bawah ini `income` milik P10 dibuat kosong.

In [9]:
from sklearn.impute import SimpleImputer

df_nan = df.copy()
df_nan.loc[df_nan['id'] == 'P10', 'income'] = None   # P10 ada di test set
X_nan = df_nan[FEATURES]

X_train_nan = X_nan.loc[X_train.index]

nilai_pengisi = pd.DataFrame({
    'sumber': ['SALAH: mean seluruh data', 'BENAR: mean data train saja'],
    'income': [
        SimpleImputer(strategy='mean').fit(X_nan).statistics_[0],
        SimpleImputer(strategy='mean').fit(X_train_nan).statistics_[0],
    ],
}).round(2).set_index('sumber')
nilai_pengisi

,income
sumber,
SALAH: mean seluruh data,63.78
BENAR: mean data train saja,72.00


Nilai kosong pada baris test diisi dengan angka yang **dihitung dari baris test lainnya**.
Di produksi hal ini mustahil: saat mengisi data pemohon baru, kita hanya punya statistik
dari data historis (train).

## 7. Seberapa besar efeknya ke skor model?

Pada data 10 baris di atas, selisihnya masih kecil. Tapi pada langkah yang lebih "rakus"
seperti **feature selection**, kebocoran bisa mengubah kesimpulan sepenuhnya.

Di bawah ini datanya **murni acak** — tidak ada hubungan sama sekali antara fitur dan
target. Skor jujurnya harus sekitar **0.5** (setara menebak).

In [10]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
X_acak = rng.normal(size=(60, 300))      # 300 fitur acak
y_acak = rng.integers(0, 2, size=60)     # target acak

cv = StratifiedKFold(5, shuffle=True, random_state=0)

# ❌ memilih 5 fitur terbaik memakai SELURUH data (termasuk fold validasi)
X_terpilih = SelectKBest(f_classif, k=5).fit_transform(X_acak, y_acak)
skor_bocor = cross_val_score(LogisticRegression(), X_terpilih, y_acak, cv=cv).mean()

# ✅ seleksi fitur di dalam pipeline -> di-fit ulang di setiap fold train
pipe = make_pipeline(SelectKBest(f_classif, k=5), LogisticRegression())
skor_benar = cross_val_score(pipe, X_acak, y_acak, cv=cv).mean()

print(f'DENGAN kebocoran : {skor_bocor:.3f}')
print(f'TANPA  kebocoran : {skor_benar:.3f}')

DENGAN kebocoran : 0.800
TANPA  kebocoran : 0.533


Akurasi **0.80 pada data yang sepenuhnya acak**. Model seperti ini akan terlihat bagus di
laporan, lalu gagal total begitu dipakai. Inilah bahaya sebenarnya dari data leakage:
bukan errornya, tapi **tidak adanya error** — masalahnya baru ketahuan di produksi.

## 8. Solusi praktis: `Pipeline`

`Pipeline` membungkus preprocessing dan model jadi satu objek. Saat `fit`, semua langkah
belajar dari data train saja; saat `predict`, semua langkah hanya men-`transform`.
Urutannya tidak mungkin tertukar.

In [11]:
model = make_pipeline(
    MinMaxScaler(),
    LogisticRegression(max_iter=1000),
)

model.fit(X_train, y_train)          # scaler + model di-fit pada train
print('prediksi test:', model.predict(X_test))
print('scaler di dalam pipeline -> min:', model[0].data_min_, ' max:', model[0].data_max_)

prediksi test: ['Low' 'Low' 'Medium']
scaler di dalam pipeline -> min: [ 35. 610.]  max: [130. 790.]


`min` dan `max` di dalam pipeline sama dengan hasil `fit` pada data train — bukan seluruh
data. Pipeline ini juga aman dipakai langsung di `cross_val_score` dan `GridSearchCV`.

## Ringkasan

| Langkah | Boleh `fit` di seluruh data? |
|---|---|
| `train_test_split` | — (justru langkah pertama) |
| Scaling (`MinMaxScaler`, `StandardScaler`) | ❌ train saja |
| Encoding (`OneHotEncoder`, `OrdinalEncoder`) | ❌ train saja |
| Imputasi (`SimpleImputer`) | ❌ train saja |
| Feature selection | ❌ train saja |
| Menghapus kolom / ganti nama kolom | ✅ boleh (tidak belajar dari data) |

Pola yang aman:

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = make_pipeline(MinMaxScaler(), LogisticRegression())
model.fit(X_train, y_train)      # semua langkah belajar dari train
model.score(X_test, y_test)      # test hanya di-transform
```

Cara cepat mengenali kebocoran: **cari `fit` atau `fit_transform` yang dipanggil sebelum
`train_test_split`, atau yang dipanggil pada `X_test`.**

## Export tabel ke PNG untuk slide

Jalankan sel di bawah kalau tabelnya mau dipakai sebagai gambar di PowerPoint.

In [12]:
import dataframe_image as dfi

dfi.export(df, 'leakage_data_original.png', table_conversion='matplotlib', dpi=200)
dfi.export(test_salah, 'leakage_test_wrong.png', table_conversion='matplotlib', dpi=200)
dfi.export(test_benar, 'leakage_test_correct.png', table_conversion='matplotlib', dpi=200)
dfi.export(perbandingan, 'leakage_comparison.png', table_conversion='matplotlib', dpi=200)
dfi.export(statistik, 'leakage_scaler_stats.png', table_conversion='matplotlib', dpi=200)
dfi.export(nilai_pengisi, 'leakage_imputer_values.png', table_conversion='matplotlib', dpi=200)